In [0]:
files = [f.name for f in dbutils.fs.ls("/Volumes/my_catalog/products_raw/products/Age-Sex_Data/")]
dbutils.widgets.dropdown("filename", files[0] if files else "", files, "Select filename")
display(dbutils.fs.ls("/Volumes/my_catalog/products_raw/products/Age-Sex_Data/"))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql.window import *

In [0]:
# Get selected filename from widget
selected_file = dbutils.widgets.get("filename")

# Validate that a file was selected
if not selected_file:
    raise ValueError("No file selected. Please select a file from the dropdown widget.")

# Construct full file path
file_path = f"/Volumes/my_catalog/products_raw/products/Age-Sex_Data/{selected_file}"

# Load CSV with proper options
df_read = (spark.read
           .format("csv")
           .option("header", "true")
           .option("inferSchema", "true")
           .load(file_path))

# Display the loaded data
display(df_read)

In [0]:
# Count total number of rows in the DataFrame
df_count = df_read.count()
display(df_count)

In [0]:
df_count_age_0 = df_read.filter((col("age") == 0) | (col("age") == 1)).count()
display(df_count_age_0)

In [0]:
df_dedup=(df_read.dropDuplicates()\
        .dropna()).count()
display(df_dedup)

In [0]:
df_window = df_read.dropDuplicates().dropna()\
    .withColumn("Row_Number", row_number().over(Window.partitionBy("age").orderBy(desc("age"))))\
    .filter(col("Row_Number") == 1)
display(df_window)

In [0]:
df_col = df_read.columns
display(df_col)

In [0]:
df_dim_year=df_read.select("year").display()


In [0]:

year_df = df_read.select("Year").distinct()

window_spec = Window.orderBy("Year")

dim_year = year_df.withColumn(
    "year_key",
    row_number().over(window_spec)
)
display(year_df)
display(dim_year)